# Pratica 1 · Introduzione all'apprendimento per rinforzo

**DIH DATalife** — Corso di Reinforcement Learning (RL) da zero

Questo notebook è la pratica guidata della lezione uno. Riprende l'ambiente del corridoio che hai programmato in classe e ti propone due esercizi per ampliarlo e per confrontare politiche.

**Prima di iniziare**:

- Esegui le celle in ordine, dall'alto verso il basso
- Ogni esercizio contiene una struttura di partenza, codice già scritto con alcune parti contrassegnate come `# TODO`
- Completa ogni `TODO` ed esegui la cella per verificare la tua soluzione
- Se ti blocchi, apri la soluzione che si trova subito sotto ogni esercizio, ma **prova prima da solo**

Questo notebook funziona su Google Colab senza installare nulla: ti serve solo `numpy`, che è già preinstallato.

## Preparazione: l'ambiente della lezione

Esegui questa cella per avere a disposizione il `PasilloEntorno`, la `politica_aleatoria` e una funzione ausiliaria (`contar_pasos_hasta_meta`) che userai nell'esercizio due. È lo stesso codice che hai visto in classe.

In [ ]:
import numpy as np


class PasilloEntorno:
    def __init__(self, longitud=5):
        self.longitud = longitud
        self.meta = longitud - 1
        self.estado = 0  # posición inicial del agente

    def reset(self):
        """Reinicia el entorno y devuelve el estado inicial."""
        self.estado = 0
        return self.estado

    def step(self, accion):
        """
        Ejecuta una acción en el entorno.
        accion: 0 = mover a la izquierda, 1 = mover a la derecha
        Devuelve: (nuevo_estado, recompensa, terminado)
        """
        if accion == 1:
            self.estado = min(self.estado + 1, self.longitud - 1)
        elif accion == 0:
            self.estado = max(self.estado - 1, 0)

        terminado = self.estado == self.meta
        recompensa = 10 if terminado else -1

        return self.estado, recompensa, terminado


def politica_aleatoria(estado, num_acciones=2):
    """Política estocástica: elige una acción al azar, sin usar el estado."""
    return np.random.randint(num_acciones)


def contar_pasos_hasta_meta(entorno, elegir_accion, max_pasos=50, semilla=None):
    """
    Ejecuta un episodio completo con la política que le pases en
    'elegir_accion' (una función que recibe el estado y devuelve una
    acción) y devuelve cuántos pasos tardó en llegar a la meta, o
    max_pasos si no llegó a tiempo. La usarás en el ejercicio dos.
    """
    if semilla is not None:
        np.random.seed(semilla)

    estado = entorno.reset()
    for paso in range(1, max_pasos + 1):
        accion = elegir_accion(estado)
        estado, recompensa, terminado = entorno.step(accion)
        if terminado:
            return paso
    return max_pasos


print("Entorno y funciones auxiliares listos.")

## Esercizio 1 · Dal corridoio a una griglia con trappola

Il corridoio della lezione ha una sola dimensione: sinistra o destra. In questo esercizio lo ampliamo a una **piccola griglia 2D** con una cella "trappola" che assegna una ricompensa negativa.

**Regole del nuovo ambiente `CuadriculaEntorno`**:

- L'agente parte dall'angolo in alto a sinistra, nella posizione (0, 0)
- L'obiettivo si trova nell'angolo in basso a destra, nella posizione (filas-1, columnas-1)
- Ci sono quattro azioni possibili: 0 = su, 1 = giù, 2 = sinistra, 3 = destra
- Se l'agente finisce sulla cella trappola, l'episodio termina lì con ricompensa -5 (non raggiunge l'obiettivo)
- Se l'agente raggiunge l'obiettivo, l'episodio termina con ricompensa +10
- In ogni altro passo, la ricompensa è -1, come nel corridoio

**Suggerimenti**:

- Lo stato ora è una tupla `(fila, columna)`, invece di un unico numero
- Per non uscire dalla griglia, usa `min` e `max` come fa `PasilloEntorno` con `self.longitud`
- Confronta sempre `self.estado` (già aggiornato) con `self.trampa` e con `self.meta` per decidere la ricompensa e se l'episodio termina

Completa i due `TODO` della cella seguente ed eseguila per verificare la tua soluzione con un episodio di esempio.

In [ ]:
class CuadriculaEntorno:
    """
    Cuadrícula de filas x columnas. El agente empieza en (0, 0) y la
    meta está en (filas-1, columnas-1). Hay una celda "trampa": si el
    agente cae en ella, recibe una recompensa negativa y el episodio
    termina ahí mismo, sin llegar a la meta.

    Acciones: 0 = arriba, 1 = abajo, 2 = izquierda, 3 = derecha
    """

    def __init__(self, filas=3, columnas=3, trampa=(1, 1)):
        self.filas = filas
        self.columnas = columnas
        self.meta = (filas - 1, columnas - 1)
        self.trampa = trampa
        self.estado = (0, 0)

    def reset(self):
        self.estado = (0, 0)
        return self.estado

    def step(self, accion):
        fila, columna = self.estado

        # TODO 1: actualiza 'fila' y 'columna' según la acción.
        # Usa min/max para no salirte de la cuadrícula, igual que
        # hace PasilloEntorno con self.longitud.
        #   0 = arriba    -> fila - 1
        #   1 = abajo     -> fila + 1
        #   2 = izquierda -> columna - 1
        #   3 = derecha   -> columna + 1

        self.estado = (fila, columna)

        # TODO 2: calcula 'recompensa' y 'terminado' según el nuevo
        # self.estado:
        #   - si self.estado es self.trampa -> recompensa -5, terminado True
        #   - si self.estado es self.meta   -> recompensa +10, terminado True
        #   - en cualquier otro caso        -> recompensa -1, terminado False
        recompensa = None
        terminado = None

        return self.estado, recompensa, terminado


def politica_aleatoria_cuadricula(estado, num_acciones=4):
    return np.random.randint(num_acciones)


def ejecutar_episodio_cuadricula(entorno, max_pasos=30, semilla=None):
    """Igual que ejecutar_episodio del pasillo, adaptado a la cuadrícula."""
    if semilla is not None:
        np.random.seed(semilla)

    estado = entorno.reset()
    recompensa_acumulada = 0

    for paso in range(1, max_pasos + 1):
        accion = politica_aleatoria_cuadricula(estado)
        nuevo_estado, recompensa, terminado = entorno.step(accion)
        recompensa_acumulada += recompensa

        print(f"Paso {paso}: {estado} -> acción={accion} -> {nuevo_estado}, recompensa={recompensa}")

        estado = nuevo_estado

        if terminado:
            if estado == entorno.trampa:
                print(f"\nCayó en la trampa en el paso {paso}")
            else:
                print(f"\n¡Meta alcanzada en el paso {paso}!")
            break

    print(f"Recompensa acumulada del episodio: {recompensa_acumulada}")
    return recompensa_acumulada


# Prueba tu código con un episodio de ejemplo
ejecutar_episodio_cuadricula(CuadriculaEntorno(), max_pasos=30, semilla=3)

<details>
<summary>💡 Vedi la soluzione</summary>

```python
class CuadriculaEntorno:
    def __init__(self, filas=3, columnas=3, trampa=(1, 1)):
        self.filas = filas
        self.columnas = columnas
        self.meta = (filas - 1, columnas - 1)
        self.trampa = trampa
        self.estado = (0, 0)

    def reset(self):
        self.estado = (0, 0)
        return self.estado

    def step(self, accion):
        fila, columna = self.estado

        # TODO 1 risolto: applichiamo lo spostamento in base all'azione,
        # limitando con min/max per non uscire dalla griglia
        if accion == 0:
            fila = max(fila - 1, 0)
        elif accion == 1:
            fila = min(fila + 1, self.filas - 1)
        elif accion == 2:
            columna = max(columna - 1, 0)
        elif accion == 3:
            columna = min(columna + 1, self.columnas - 1)

        self.estado = (fila, columna)

        # TODO 2 risolto: confrontiamo il nuovo stato con la trappola e
        # con l'obiettivo per decidere ricompensa e se l'episodio termina
        if self.estado == self.trampa:
            recompensa = -5
            terminado = True
        elif self.estado == self.meta:
            recompensa = 10
            terminado = True
        else:
            recompensa = -1
            terminado = False

        return self.estado, recompensa, terminado
```

**Perché funziona così**: lo schema è lo stesso di `PasilloEntorno`, solo che ora hai due coordinate invece di una. Controlla prima la trappola e poi l'obiettivo (in quell'ordine) perché, in griglie più grandi, potrebbero presentarsi scenari in cui l'ordine dei controlli conta; qui, con `trampa` e `meta` in celle diverse, l'ordine non cambia il risultato, ma è buona abitudine mantenerlo esplicito.
</details>

## Esercizio 2 · Politica greedy contro politica casuale

In classe hai visto solo la politica casuale: non usa alcuna informazione sullo stato e sceglie ogni azione a caso. In questo esercizio implementerai una **politica greedy** (*greedy*, avida): una politica deterministica e molto semplice che, siccome conosce in anticipo dove si trova l'obiettivo, si muove sempre verso di esso.

Lavorerai sul `PasilloEntorno` della lezione (e non sulla griglia dell'esercizio uno), così potrai confrontare entrambe le politiche nello stesso ambiente.

**Suggerimenti**:

- Lo stato del corridoio è un numero compreso tra zero e la posizione dell'obiettivo
- Se lo stato si trova prima dell'obiettivo, l'azione corretta è muoversi a destra (1)
- Se lo stato si trova dopo l'obiettivo, l'azione corretta è muoversi a sinistra (0); in questo ambiente specifico non dovrebbe mai succedere, ma è bene prevederlo
- Usa `contar_pasos_hasta_meta`, già definita nella cella di preparazione, per misurare quanti passi impiega ogni politica a raggiungere l'obiettivo

Completa il `TODO` di `politica_codiciosa` ed esegui la cella per confrontare i passi medi di venti episodi con ciascuna politica.

In [ ]:
entorno_pasillo = PasilloEntorno()

# Referencia: cuántos pasos tarda la política aleatoria del capítulo,
# medidos en veinte episodios
pasos_aleatoria = [
    contar_pasos_hasta_meta(entorno_pasillo, politica_aleatoria, max_pasos=50)
    for _ in range(20)
]


def politica_codiciosa(estado):
    """
    Política determinista: siempre se mueve hacia la meta. Como
    conoces la posición de la meta de antemano, no hace falta
    explorar para decidir la acción.
    """
    meta = entorno_pasillo.meta

    # TODO: devuelve 1 (derecha) si 'estado' está antes de la meta,
    # o 0 (izquierda) si 'estado' está después de la meta.


pasos_codiciosa = [
    contar_pasos_hasta_meta(entorno_pasillo, politica_codiciosa, max_pasos=50)
    for _ in range(20)
]

print("Media de pasos - política aleatoria:", np.mean(pasos_aleatoria))
print("Media de pasos - política codiciosa:", np.mean(pasos_codiciosa))

<details>
<summary>💡 Vedi la soluzione</summary>

```python
def politica_codiciosa(estado):
    meta = entorno_pasillo.meta

    # TODO risolto: confrontiamo lo stato con l'obiettivo e scegliamo
    # l'azione che avvicina l'agente
    if estado < meta:
        return 1  # destra
    elif estado > meta:
        return 0  # sinistra
    else:
        return 1  # per sicurezza, anche se non dovrebbe essere chiamata già sull'obiettivo
```

**Cosa dovresti osservare**: la politica casuale impiega in media molti passi, con grande variabilità da un episodio all'altro (a volte arriva rapidamente, altre volte gira a vuoto). La politica greedy, conoscendo in anticipo dove si trova l'obiettivo, impiega sempre lo stesso numero di passi: la distanza iniziale dall'obiettivo, senza passi in più.

Questo confronto è proprio il punto di partenza dei prossimi capitoli: la politica greedy di qui "bara" perché conosce già in anticipo la posizione dell'obiettivo. Un agente reale quasi mai dispone di questa informazione: per questo serve un algoritmo, come il Q-learning, che la impara dall'esperienza invece di fornirtela già risolta.
</details>

## Per continuare a esercitarti

Se ti è avanzato del tempo, riprendi le proposte del manuale:

- Aggiungi una terza azione, per esempio "restare fermo", sia in `PasilloEntorno` sia in `CuadriculaEntorno`
- Cambia il disegno della ricompensa (per esempio, 0 per passo invece di -1) e osserva se l'agente ha ancora incentivo a raggiungere rapidamente l'obiettivo
- Combina i due esercizi: prova una politica greedy su `CuadriculaEntorno` che eviti la trappola di proposito

Salva i tuoi risultati: li riutilizzerai come punto di confronto quando, nelle prossime lezioni, sostituirai queste politiche con una che impara davvero, come il Q-learning.